In [ ]:
#Use rclone to clone google drive files
#see misc_files/ssh_info.txt

#compare with meterikc in cigale 

#find cigale spectra at: 16 models,  
# /d/vel2/ddale/phangs/synthetic/templates/00_best_model.fits  -  spectrum
# /d/vel2/ddale/phangs/synthetic/templates/models-block-0.fits   - filter flux 

from ImageScience import ImageScience
from SpecScience import SpecScience
from Functions import *
from matplotlib.patches import Circle
from scipy.ndimage import rotate as ndimage_rotate

import glob
locations = [[202.5062429, 47.2143358], [202.4335225, 47.1729608], [202.4340450, 47.1732517], [202.4823742, 47.1958589]]
loc_sky = SkyCoord(ra=locations[0][0] * u.deg, dec=locations[0][1] * u.deg)
radius = 0.75*u.arcsec
image_files = glob.glob('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/*anchor.fits')
im = ImageScience()
for image in image_files:
    im.load_image(extract_filter_name(image), image)
filters = [extract_filter_name(x) for x in image_files]
image_files
x = SpecScience()
d = '/project/galaxies/tjuchau/data_files/JWST/miri_mrs/v2_jun2026'
files = glob.glob(d+'/*')
nir_files = glob.glob('/project/galaxies/tjuchau/data_files/JWST/nirspec/v2_jun2026/ne*')
files = np.concat([files,nir_files])
cubes = []
for i, file in enumerate(files):
    if 'miri_mrs' in file:
        channel = file.split('/')[-1].split('_ch')[-1][0]
        idx = file.split('/')[-1].split('_ch')[-1][2]
        name = f'ch{channel}_{idx}'
    elif 'f290lp' in file:
        name = 'g_3'
    elif 'f170lp' in file:
        name = 'g_2'
    elif 'f100lp' in file:
        name = 'g_1'
    x.load_cube(name, file)
    cubes.append(name)
files

In [ ]:
for i, cube in enumerate(cubes):
        filters_in = x.which_filters(x.cubes[cube], filters, must_all=True)
        for fil in filters_in:
                x.align_to_image(im, fil, cube, fil, out_name = fil, out_file = None, 
                        fwhm = 2.3, detection_threshold = 2.0, max_offset_pixels = 10.0, min_matches = 2, show_diagnostic_plots = True)
                x.create_synthetic_image(fil, fil,out_name=f'synth_{fil}')
                x.display([f'synth_{fil}'], loc_sky, radius, show_grid=True)
                im.display(fil, loc_sky, radius, show_grid)
        if cube == cubes[-1]:
                continue
        else:
                x.append_cubes([cubes[i], cubes[i+1]], out_name=f'{cubes[i]}_{cubes[i+1]}')
for fil in filters:
        if fil in x.imag
for cube in x.cubes.keys():
        x.get_spectrum(cube, loc_sky, 0.75*u.arcsec, background_annulus_thickness=0,
                buffer=0*u.arcsec, replace_negatives = False, aperture_type='cyl', out_name=cube)

x.stitch_spectra(cubes, anchor_idx=0, method='mean', out_name='full')
x.append_cubes(['g_1', "g_2"], out_name='g12')
for fil in filters:
        x.apply_filter('full', fil, out_name=fil)

In [ ]:
phot_flux = im.get_background_subtracted_flux('F187N', loc_sky, 0.75*u.arcsec, background_annulus_thickness=0)['source_flux']
synth_flux = x.apply_filter('full', 'F187N')
print(f'synth_flux:{synth_flux}\nPhoto_flux:{phot_flux}\nratio:{synth_flux/phot_flux}')
x.stitch_spectra(['g_1', 'g_2'], method='add_shift', out_name='test')
x.stitch_spectra(['g_1', 'g_2'], anchor_idx=1, method='add_shift', out_name='test1')
test = x.apply_filter('test', 'F187N')
test1 = x.apply_filter('test1', 'F187N')
print(test)
print(test/phot_flux)
print(test1)
print(test1/phot_flux)

In [ ]:
wl, trans = get_filter_data('F187N')
plt.plot(wl, trans*np.max(x.spectra['test']['F_nu']))
x.stitch_spectra(['g_1', 'g_2'], method='add_shift', out_name='test')
x.stitch_spectra(['g_1', 'g_2'], anchor_idx=1, method='add_shift', out_name='test1')
x.stitch_spectra(['g_1', 'g_2'], anchor_idx=1, method='mean', out_name='test2')
mask = (x.spectra['test']['wavelength'] > 1.86*u.um) & (x.spectra['test']['wavelength'] < 1.925*u.um)
mask1 = (x.spectra['test1']['wavelength'] > 1.86*u.um) & (x.spectra['test1']['wavelength'] < 1.925*u.um)
mask2 = (x.spectra['test2']['wavelength'] > 1.86*u.um) & (x.spectra['test2']['wavelength'] < 1.925*u.um)

plt.plot(x.spectra['test']['wavelength'][mask], x.spectra['test']['F_nu'][mask], color='red', 
    label=f'{x.apply_filter('test', 'F187N')/phot_flux}')
plt.plot(x.spectra['test2']['wavelength'][mask2], x.spectra['test2']['F_nu'][mask2], color='green', 
    label=f'{x.apply_filter('test2', 'F187N')/phot_flux}')
plt.plot(x.spectra['test1']['wavelength'][mask1], x.spectra['test1']['F_nu'][mask1], color='blue',
    label=f'{x.apply_filter('test1', 'F187N')/phot_flux}')
plt.legend()
plt.show()


In [ ]:
for cube in cubes[::-1]:
    plt.plot(x.spectra[cube]['wavelength'], x.spectra[cube]['F_nu'], label = cube)
plt.plot(x.spectra['full']['wavelength'], x.spectra['full']['F_nu'], lw=0.5, color = 'black', label = 'stitched')
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor = (1,1))
plt.show()


In [ ]:
for cube in im.cubes:
    print(cube.shape)

In [ ]:
obj = x
image_obj = im
specs = []
reallign_cubes=True
    for cube in cubes:
        filter_in = x.which_filters(cube, filter_list)
        for fil in filters_in:
            obj.align_to_image(image_obj, fil, obj.cubes[cube], fil, out_name = cube, out_file = None, 
                fwhm = 2.3, detection_threshold = 5.0, max_offset_pixels = 10.0, min_matches = 3, show_diagnostic_plots = True)
loc_sky = SkyCoord(ra=loc[0] * u.deg, dec=loc[1] * u.deg)
show_real_images = ['F150W', 'F187N']
show_synth_images = ['test_150', 'test_187']
obj.create_synthetic_image('g12', 'F150W', warnings=True, counter='energy', out_name='test_150')
obj.create_synthetic_image('g12', 'F187N', warnings=True, counter='energy', out_name='test_187')
radius = 0.75*u.arcsec
# TJ create figure axes, fontsizes, marker sizes, colors, initial axis limits, etc
fig = plt.figure(figsize=(45, 30))
ax_spec = fig.add_axes((0.05, 0.4, 1, 0.6))
ax_scat = fig.add_axes((0.05, 0.05, 1, 0.35))
fontsize_sm = 35
fontsize_lg = 45
marker_size = 250
cube_colors = ['purple', 'blue', 'cyan', 'green', 'orange', 'red', 'pink','purple', 'blue', 'cyan', 'green', 'orange', 'red', 'pink', 'purple']
spec_y_min = 1  # TJ Flux should always be around 10^-20 so setting limits of 0-1 should never be too strict
spec_y_max = 0

# TJ generate tick labels and sizes
ax_scat.tick_params(axis='x', which='minor', width=2, length=10, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.tick_params(axis='x', which='major', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.tick_params(axis='y', which='both', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.set_xlabel('wavelength (m)', fontsize=40)
ax_scat.set_ylabel('synthetic/photometric flux', fontsize=40)
ax_spec.tick_params(axis='x', which='minor', width=2, length=10, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_spec.tick_params(axis='x', which='major', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_spec.tick_params(axis='y', which='both', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
units = obj.spectra['g_1']['F_nu'].unit
ax_spec.set_ylabel(f'F_nu {units}', fontsize=40)

ax_spec.set_title(f"{radius} location 0",
                    fontsize=50)

# TJ set scale to logorithmic on both horizontal axes and the vertical axis
ax_scat.set_xscale('log')
ax_spec.set_xscale('log')
ax_spec.set_yscale('log')

for i,name in enumerate(names):
    if i == len(names)-1:
        continue
    first_cube = obj.spectra[names[i]]
    second_cube = obj.spectra[names[i+1]]
    end_overlap = first_cube['wavelength'][-1]
    start_overlap = second_cube['wavelength'][0]
    mask = (obj.spectra['full']['wavelength'] > start_overlap) & (obj.spectra['full']['wavelength'] < end_overlap)
    ax_spec.plot(obj.spectra['full']['wavelength'][mask], obj.spectra['full']['F_nu'][mask], color = 'black', lw=0.8)
    ax_spec.plot(obj.spectra[names]['wavelength'], obj.spectra[names]['F_nu'], color = cube_colors[i], lw=5)
ax_spec.plot(obj.spectra[names[-1]]['wavelength'], obj.spectra[names[-1]]['F_nu'], alpha=0.5, color=cube_colors[-1], linewidth=5)

# TJ initialize filter name label locations
label_positions = []
ax_spec.scatter([], [], marker='*', s=marker_size, color='black', label='Synth')
ax_spec.scatter([], [], marker="o", s=marker_size, color='black', label='Photo')

for i, fil in enumerate(filters):
    filter_wl, filter_trans, eff_width, pivot_wl, mean_wl = get_filter_data(fil, aux_info=True)
    ax_spec.scatter(mean_wl, obj.data[fil], marker='*', s=marker_size, color = 'black')
    ax_scat.plot(obj.spectra['full']['wavelength'], [1] * len(obj.spectra['full']['wavelength']), color='white', alpha=0)
    phot_flux = image_obj.get_background_subtracted_flux(fil, loc_sky, radius, background_annulus_thickness=0)['source_flux']
    ax_spec.scatter(mean_wl, phot_flux, marker="o", s=marker_size, color='black')
    ax_spec.hlines(y=phot_flux.value, xmin=(mean_wl - eff_width/2).value, xmax=(mean_wl + eff_width/2).value, color='black',
                       alpha=0.7, linewidth=3)
    ratio = obj.data[fil]/phot_flux
    ax_scat.scatter(mean_wl, ratio, s=marker_size, color='black')
    ax_scat.hlines(y=ratio, xmin=(mean_wl - eff_width/2).value, xmax=(mean_wl + eff_width/2).value, color='black', alpha=0.7, linewidth=3)

    # TJ default offset is 0.05 lower than the scatter point
    y_offset = -0.05

    # TJ transform the entire figure axes to a coordinate system to calculate distances between labels
    x_disp, y_disp = ax_spec.transData.transform((mean_wl.value, ratio))

    # TJ check if labels are too close to another label
    too_close = False
    for (xx, yy) in label_positions:
        if abs(x_disp - xx) < 10 and abs(y_disp + y_offset - yy) < 20:
            #TJ labels are vertical, so check for wider overlap in y-pixel space
            too_close = True
            break
    
        # TJ if overlapping, nudge upward instead of downward
        if too_close:
            y_offset = +0.2
            # TJ if the location of the scatter point is less than 0.8, always default to labeling it up instead of down
        if ratio < 0.8:
            y_offset = +0.4
        # TJ save adjusted label position
        label_positions.append((x_disp, y_disp + y_offset))

        # TJ this filter is extremely close to another filter and should always be above instead of below
        #if name == "F182M":
        #    y_offset = +0.25
        # TJ same with this one
        #if name == 'F212N':
        #    y_offset = +0.25

        # TJ now plot text in chosen coordinates
    ax_scat.text(
        mean_wl.value, ratio + y_offset,
        fil+f'{ratio:.2f}',
        ha="center", va="top",
        fontsize=fontsize_sm, rotation=90, color='black',
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=0.5)
    )

image_locations = [(0.05, 0.75, 0.2, 0.2),
    (0.3, 0.75, 0.2, 0.2), 
    (0.6, 0.41, 0.2, 0.2), 
    (0.8, 0.41, 0.2, 0.2),
    (0.85, 0.65, 0.18, 0.18)]
    

for idx in [0, 1]:
    real_pix_size  = image_obj.get_pix_scale(show_real_images[idx]).to_value(u.arcsec)
    synth_pix_size = obj.get_pix_scale(show_synth_images[idx]).to_value(u.arcsec)
    raw_synth = obj.images[show_synth_images[idx]]
    nan_mask = np.isnan(raw_synth)
    synth_full_clean = raw_synth.copy()
    synth_full_clean[nan_mask] = 0.0

    # Compute rotation angle before doing anything with the data
    real_wcs  = image_obj.wcs[show_real_images[idx]]
    synth_wcs = obj.wcs[show_synth_images[idx]]
    real_cd   = real_wcs.pixel_scale_matrix
    synth_cd  = synth_wcs.pixel_scale_matrix
    real_pa   = np.degrees(np.arctan2(-real_cd[0, 1],  real_cd[1, 1]))
    synth_pa  = np.degrees(np.arctan2(-synth_cd[0, 1], synth_cd[1, 1]))
    delta_pa  = synth_pa - real_pa
    synth_full_rotated = ndimage_rotate(synth_full_clean, delta_pa, reshape=False, order=3, cval=np.nan)
    pc = synth_wcs.wcs.get_pc()  # 2x2 rotation matrix
    cdelt = synth_wcs.wcs.cdelt  # pixel scales

    angle_rad = np.radians(-delta_pa)
    rot_matrix = np.array([
        [ np.cos(angle_rad), np.sin(angle_rad)],
        [-np.sin(angle_rad), np.cos(angle_rad)]
    ])

    synth_wcs_rotated = synth_wcs.deepcopy()
    synth_wcs_rotated.wcs.pc = pc @ rot_matrix
    # Make sure no CD matrix is set, which would override PC
    if hasattr(synth_wcs_rotated.wcs, 'cd'):
        del synth_wcs_rotated.wcs.cd
    synth_wcs_rotated.wcs.set()  # recompute internal state

    # Now do the cutout on the rotated image with the corrected WCS
    cutout_real = Cutout2D(
        image_obj.images[show_real_images[idx]],
        position=loc_sky,
        size=(radius * 3, radius * 3),
        wcs=real_wcs
    )
    cutout_synth = Cutout2D(
        synth_full_rotated,
        position=loc_sky,
        size=(radius * 3, radius * 3),
        wcs=synth_wcs_rotated
    )

    real_data  = cutout_real.data.copy()
    synth_data = cutout_synth.data.copy()  # already a plain array, no .value needed

    # Compute norms
    real_pixels  = real_data[np.isfinite(real_data)]
    synth_pixels = synth_data[np.isfinite(synth_data)]
    med_r, sig_r = np.nanmedian(real_pixels), np.nanstd(real_pixels)
    med_s, sig_s = np.nanmedian(synth_pixels), np.nanstd(synth_pixels)
    norm_real  = colors.Normalize(vmin=med_r - 3*sig_r, vmax=med_r + 20*sig_r)
    norm_synth = colors.Normalize(vmin=med_s - 3*sig_s, vmax=med_s + 20*sig_s)

    real_radius_pix  = (radius / real_pix_size).value
    synth_radius_pix = (radius / synth_pix_size).value

    y_r, x_r = np.array(real_data.shape)  // 2
    y_s, x_s = np.array(synth_data.shape) // 2

    ax_real = fig.add_axes(image_locations[idx * 2], projection=cutout_real.wcs)
    ax_syn  = fig.add_axes(image_locations[idx * 2 + 1], projection=cutout_synth.wcs)

    cmap = plt.get_cmap("viridis").copy()
    cmap.set_under("black")
    cmap.set_over("white")

    im0 = ax_real.imshow(real_data,  origin="lower", cmap=cmap, norm=norm_real)
    im1 = ax_syn.imshow(synth_data,  origin="lower", cmap=cmap, norm=norm_synth)
    
    ax_real.coords.grid(color='white', linestyle='--', linewidth=1, alpha=0.7)
    ax_syn.coords.grid(color='white', linestyle='--', linewidth=1, alpha=0.7)

    cbar0 = plt.colorbar(im0, ax=ax_real, fraction=0.046, pad=0.04)
    cbar0.set_label("W/m2/Hz/sr", fontsize=fontsize_lg)
    cbar0.ax.tick_params(labelsize=fontsize_sm)

    cbar1 = plt.colorbar(im1, ax=ax_syn, fraction=0.046, pad=0.04)
    cbar1.set_label("W/m2/Hz/sr", fontsize=fontsize_lg)
    cbar1.ax.tick_params(labelsize=fontsize_sm)

    ax_real.add_patch(Circle((x_r, y_r), real_radius_pix,  edgecolor='red', facecolor='none', linewidth=2))
    ax_syn.add_patch( Circle((x_s, y_s), synth_radius_pix, edgecolor='red', facecolor='none', linewidth=2))

    ax_real.set_xticks([])
    ax_real.set_yticks([])
    ax_syn.set_xticks([])
    ax_syn.set_yticks([])

ymin, ymax = ax_scat.get_ylim()
text_y_pos = ymin * 1.1
ax_scat.axhline(y=1, color='gray', linestyle='--', linewidth=4, alpha=0.5)
ax_scat.axvline(x=5e-6, color='gray', linestyle='--', linewidth=4, alpha=0.7)
ax_scat.text(4.9e-6, text_y_pos, "← NIRCam",
                ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
                fontsize=fontsize_lg)

# Add MIRI label to the right
ax_scat.text(5.1e-6, text_y_pos, "MIRI →",
                ha='left', va='center',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
                fontsize=fontsize_lg)
ax_spec.legend(loc='upper right', fontsize=fontsize_lg)

plt.show()


In [ ]:
radii = np.array([0.75, 1, 1.2, 1.5])*u.arcsec
for fil in filters:
    wl, trans = get_filter_data(fil)
    if wl[0] > x.spectra['full']['wavelength'][0]:
        for radius in radii:
            x.adjust_spectrum('full', fil, im, fil, [202.5062332,47.2143345], 0.75*u.arcsec, adjustment_operation = 'add', out_name=f'{fil}_{radius}_add')
            x.adjust_spectrum('full', fil, im, fil, [202.5062332,47.2143345], 0.75*u.arcsec, adjustment_operation = 'multiply', out_name=f'{fil}_{radius}_mult')

In [ ]:
for key in x.spectra.keys():
    try:
        print(key, ": ", x.spectra[key]['correction'])
    except:
        print()

In [ ]:
'''loc0.create_synthetic_image('ch1ch2', 'F150W', warnings=True, counter='energy', out_name='synth_F150W')
loc0.create_synthetic_image('test', 'F150W', warnings=True, counter='energy', out_name='testing')

images.display(['F150W', "F187N"], locations[0], 0.75*u.arcsec, show_grid=True)
loc0.display(['synth_F150W', "testing"], locations[0], 0.75*u.arcsec, show_grid=True)'''


In [ ]:
image_files = glob.glob('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/*anchor.fits')
images = ImageScience()
for file in image_files:
    images.load_image(extract_filter_name(file), file)
filters = [extract_filter_name(x) for x in image_files]
files = glob.glob('/project/galaxies/tjuchau/data_files/JWST/nirspec/karin_reduction_v1_oct2024/*o004*.fits')
files = np.concatenate([files, glob.glob('/project/galaxies/tjuchau/data_files/JWST/miri_mrs/karin_reduction/Arm1*.fits')])

redo_stitching = True #TJ This takes several minutes
redo_correction = True
redo_alignment = True
loc0 = SpecScience()
channels = []
aligned_channels = []
anchor_filters = ['F150W', 'F200W', 'F444W', 'F560W', 'F1000W', 'F1500W', "F2100W"]
for i, file in enumerate(files):
    loc0.load_cube(f'ch{i+1}', file)
    if redo_alignment:
        try:
            loc0.align_to_image(
                images,
                anchor_filters[i],
                f'ch{i+1}',
                anchor_filters[i],
                out_name = f'aligned_ch{i+1}',
                out_file = None,
                fwhm = 2.3,
                detection_threshold = 5.0,
                max_offset_pixels = 10.0,
                min_matches = 3,
                show_diagnostic_plots = False,
            )
        except:
            try:
                print('trying lower detection threshold...')
                loc0.align_to_image(
                    images,
                    anchor_filters[i],
                    f'ch{i+1}',
                    anchor_filters[i],
                    out_name = f'aligned_ch{i+1}',
                    out_file = None,
                    fwhm = 2.3,
                    detection_threshold = 2.0,
                    max_offset_pixels = 10.0,
                    min_matches = 3,
                    show_diagnostic_plots = False,
                )
            except:
                print('trying with just a single source...')
                loc0.align_to_image(
                    images,
                    anchor_filters[i],
                    f'ch{i+1}',
                    anchor_filters[i],
                    out_name = f'aligned_ch{i+1}',
                    out_file = None,
                    fwhm = 2.3,
                    detection_threshold = 2.0,
                    max_offset_pixels = 10.0,
                    min_matches = 1,
                    show_diagnostic_plots = False,
                )

    channels.append(f'ch{i+1}')
    aligned_channels.append(f'aligned_ch{i+1}')
if redo_stitching:
    for i in range(1,8):
        loc0.get_spectrum(f'ch{i}', locations[0], 0.75*u.arcsec,0)
        loc0.get_spectrum(f'aligned_ch{i}', locations[0], 0.75*u.arcsec,0)
        if i==7:
            break
        
        loc0.append_cubes([f'ch{i}', f'ch{i+1}'])
        loc0.append_cubes([f'aligned_ch{i}', f'aligned_ch{i+1}'], out_name=f'aligned_ch{i}ch{i+1}')
    loc0.stitch_spectra(channels, method='add_shift', out_name='add_offset')
    loc0.stitch_spectra(channels, method='mult_shift', out_name='mult_offset')
    loc0.stitch_spectra(channels, method='mean', out_name='mean')
    loc0.stitch_spectra(aligned_channels, method='mean', out_name='aligned_mean')
    for fil in filters:
        needed_channels = loc0.which_channels(fil)
        if len(needed_channels)>1:
            loc0.apply_filter('mean', fil, out_name=fil)
            loc0.apply_filter('aligned_mean', fil, out_name=f'aligned_{fil}')
        else:
            loc0.apply_filter(needed_channels[0], fil, out_name=fil)
            loc0.apply_filter(f'aligned_{needed_channels[0]}', fil, out_name=f'aligned_{fil}')
    loc0.save_science('/project/galaxies/tjuchau/data_files/M51/Custom_objects/loc0.pkl')
if redo_correction:
    for i in range(1,8):
        best_filter = loc0.get_largest_filter(f'ch{i}', filters)
        loc0.adjust_spectrum(f'ch{i}', best_filter, images, best_filter, locations[0], 0.75*u.arcsec, adjustment_operation = 'add', out_name=f'adjusted_ch{i}')
        
        loc0.adjust_spectrum(f'aligned_ch{i}', best_filter, images, best_filter, locations[0], 0.75*u.arcsec, adjustment_operation = 'add', out_name=f'aligned_adjusted_ch{i}')
    loc0.stitch_spectra(aligned_channels, method='mean', out_name='mean')

    loc0.save_science('/project/galaxies/tjuchau/data_files/M51/Custom_objects/loc0.pkl')

loc0.load_science('/project/galaxies/tjuchau/data_files/M51/Custom_objects/loc0.pkl')

print('Done loading data')

In [ ]:
obj = x
image_obj = images
loc_sky = SkyCoord(ra=locations[0][0] * u.deg, dec=locations[0][1] * u.deg)
show_real_images = ['F150W', 'F187N']
show_synth_images = ['test_150', 'test_187']
obj.create_synthetic_image('aligned_ch1ch2', 'F150W', warnings=True, counter='energy', out_name='test_150')
obj.create_synthetic_image('aligned_ch1ch2', 'F187N', warnings=True, counter='energy', out_name='test_187')
radius = 0.75*u.arcsec
# TJ create figure axes, fontsizes, marker sizes, colors, initial axis limits, etc
fig = plt.figure(figsize=(45, 30))
ax_spec = fig.add_axes((0.05, 0.4, 1, 0.6))
ax_scat = fig.add_axes((0.05, 0.05, 1, 0.35))
fontsize_sm = 35
fontsize_lg = 45
marker_size = 250
cube_colors = ['purple', 'blue', 'cyan', 'green', 'orange', 'red', 'pink']
spec_y_min = 1  # TJ Flux should always be around 10^-20 so setting limits of 0-1 should never be too strict
spec_y_max = 0

# TJ generate tick labels and sizes
ax_scat.tick_params(axis='x', which='minor', width=2, length=10, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.tick_params(axis='x', which='major', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.tick_params(axis='y', which='both', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_scat.set_xlabel('wavelength (m)', fontsize=40)
ax_scat.set_ylabel('synthetic/photometric flux', fontsize=40)
ax_spec.tick_params(axis='x', which='minor', width=2, length=10, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_spec.tick_params(axis='x', which='major', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
ax_spec.tick_params(axis='y', which='both', width=3, length=15, right=True, top=True, direction='in',
                    labelsize=fontsize_sm)
units = obj.spectra['ch1']['F_nu'].unit
ax_spec.set_ylabel(f'F_nu {units}', fontsize=40)

ax_spec.set_title(f"{radius} location 0",
                    fontsize=50)

# TJ set scale to logorithmic on both horizontal axes and the vertical axis
ax_scat.set_xscale('log')
ax_spec.set_xscale('log')
ax_spec.set_yscale('log')

for i in range(1,7):
    first_cube = obj.spectra[f'ch{i}']
    second_cube = obj.spectra[f'ch{i+1}']
    start_overlap = first_cube['wavelength'][-1]
    end_overlap = second_cube['wavelength'][0]
    mask = (obj.spectra['mean']['wavelength'] > start_overlap) & (obj.spectra['mean']['wavelength'] < end_overlap)
    ax_spec.plot(obj.spectra['mean']['wavelength'][mask], obj.spectra['mean']['F_nu'][mask], color = 'black', lw=0.8)
    ax_spec.plot(obj.spectra[f'aligned_ch{i}']['wavelength'], obj.spectra[f'aligned_ch{i}']['F_nu'], alpha=0.5, color=cube_colors[i-1], linewidth=5)
    ax_spec.plot(obj.spectra[f'aligned_adjusted_ch{i}']['wavelength'], obj.spectra[f'aligned_adjusted_ch{i}']['F_nu'], alpha=0.5, color=cube_colors[i-1], linewidth=5, linestyle = '--')
ax_spec.plot(obj.spectra['ch7']['wavelength'], obj.spectra['ch7']['F_nu'], alpha=0.5, color=cube_colors[6], linewidth=5)

# TJ initialize filter name label locations
label_positions = []
ax_spec.scatter([], [], marker='*', s=marker_size, color='black', label='Synth')
ax_spec.scatter([], [], marker="o", s=marker_size, color='black', label='Photo')

for fil in filters:
    filter_wl, filter_trans, eff_width, pivot_wl, mean_wl = get_filter_data(fil, aux_info=True)
    ax_spec.scatter(mean_wl, obj.data[fil], marker='*', s=marker_size, color = 'black')
    ax_scat.plot(obj.spectra['mean']['wavelength'], [1] * len(obj.spectra['mean']['wavelength']), color='white', alpha=0)
    phot_flux = images.get_background_subtracted_flux(fil, locations[0], radius, background_annulus_thickness=0)['source_flux']
    ax_spec.scatter(mean_wl, phot_flux, marker="o", s=marker_size, color='black')
    ax_spec.hlines(y=phot_flux.value, xmin=(mean_wl - eff_width/2).value, xmax=(mean_wl + eff_width/2).value, color='black',
                       alpha=0.7, linewidth=3)
    ratio = obj.data[fil]/phot_flux
    new_ratio = obj.data[fil]/phot_flux
    ax_scat.scatter(mean_wl, ratio, s=marker_size, color='black')
    ax_scat.hlines(y=ratio, xmin=(mean_wl - eff_width/2).value, xmax=(mean_wl + eff_width/2).value, color='black', alpha=0.7, linewidth=3)

    # TJ default offset is 0.05 lower than the scatter point
    y_offset = -0.05

    # TJ transform the entire figure axes to a coordinate system to calculate distances between labels
    x_disp, y_disp = ax_spec.transData.transform((mean_wl.value, ratio))

    # TJ check if labels are too close to another label
    too_close = False
    for (xx, yy) in label_positions:
        if abs(x_disp - xx) < 10 and abs(y_disp + y_offset - yy) < 20:
            #TJ labels are vertical, so check for wider overlap in y-pixel space
            too_close = True
            break
    
        # TJ if overlapping, nudge upward instead of downward
        if too_close:
            y_offset = +0.2
            # TJ if the location of the scatter point is less than 0.8, always default to labeling it up instead of down
        if ratio < 0.8:
            y_offset = +0.4
        # TJ save adjusted label position
        label_positions.append((x_disp, y_disp + y_offset))

        # TJ this filter is extremely close to another filter and should always be above instead of below
        #if name == "F182M":
        #    y_offset = +0.25
        # TJ same with this one
        #if name == 'F212N':
        #    y_offset = +0.25

        # TJ now plot text in chosen coordinates
        ax_scat.text(
            mean_wl.value, ratio + y_offset,
            name,
            ha="center", va="top",
            fontsize=fontsize_sm, rotation=90, color=color,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=0.5)
        )

image_locations = [(0.05, 0.75, 0.2, 0.2),
    (0.3, 0.75, 0.2, 0.2), 
    (0.6, 0.41, 0.2, 0.2), 
    (0.8, 0.41, 0.2, 0.2),
    (0.85, 0.65, 0.18, 0.18)]
    

for idx in [0, 1]:
    real_pix_size  = image_obj.get_pix_scale(show_real_images[idx]).to_value(u.arcsec)
    synth_pix_size = obj.get_pix_scale(show_synth_images[idx]).to_value(u.arcsec)
    raw_synth = obj.images[show_synth_images[idx]]
    nan_mask = np.isnan(raw_synth)
    synth_full_clean = raw_synth.copy()
    synth_full_clean[nan_mask] = 0.0

    # Compute rotation angle before doing anything with the data
    real_wcs  = image_obj.wcs[show_real_images[idx]]
    synth_wcs = obj.wcs[show_synth_images[idx]]
    real_cd   = real_wcs.pixel_scale_matrix
    synth_cd  = synth_wcs.pixel_scale_matrix
    real_pa   = np.degrees(np.arctan2(-real_cd[0, 1],  real_cd[1, 1]))
    synth_pa  = np.degrees(np.arctan2(-synth_cd[0, 1], synth_cd[1, 1]))
    delta_pa  = synth_pa - real_pa
    synth_full_rotated = ndimage_rotate(synth_full_clean, delta_pa, reshape=False, order=3, cval=np.nan)
    pc = synth_wcs.wcs.get_pc()  # 2x2 rotation matrix
    cdelt = synth_wcs.wcs.cdelt  # pixel scales

    angle_rad = np.radians(-delta_pa)
    rot_matrix = np.array([
        [ np.cos(angle_rad), np.sin(angle_rad)],
        [-np.sin(angle_rad), np.cos(angle_rad)]
    ])

    synth_wcs_rotated = synth_wcs.deepcopy()
    synth_wcs_rotated.wcs.pc = pc @ rot_matrix
    # Make sure no CD matrix is set, which would override PC
    if hasattr(synth_wcs_rotated.wcs, 'cd'):
        del synth_wcs_rotated.wcs.cd
    synth_wcs_rotated.wcs.set()  # recompute internal state

    # Now do the cutout on the rotated image with the corrected WCS
    cutout_real = Cutout2D(
        image_obj.images[show_real_images[idx]],
        position=loc_sky,
        size=(radius * 3, radius * 3),
        wcs=real_wcs
    )
    cutout_synth = Cutout2D(
        synth_full_rotated,
        position=loc_sky,
        size=(radius * 3, radius * 3),
        wcs=synth_wcs_rotated
    )

    real_data  = cutout_real.data.copy()
    synth_data = cutout_synth.data.copy()  # already a plain array, no .value needed

    # Compute norms
    real_pixels  = real_data[np.isfinite(real_data)]
    synth_pixels = synth_data[np.isfinite(synth_data)]
    med_r, sig_r = np.nanmedian(real_pixels), np.nanstd(real_pixels)
    med_s, sig_s = np.nanmedian(synth_pixels), np.nanstd(synth_pixels)
    norm_real  = colors.Normalize(vmin=med_r - 3*sig_r, vmax=med_r + 20*sig_r)
    norm_synth = colors.Normalize(vmin=med_s - 3*sig_s, vmax=med_s + 20*sig_s)

    real_radius_pix  = (radius / real_pix_size).value
    synth_radius_pix = (radius / synth_pix_size).value

    y_r, x_r = np.array(real_data.shape)  // 2
    y_s, x_s = np.array(synth_data.shape) // 2

    ax_real = fig.add_axes(image_locations[idx * 2], projection=cutout_real.wcs)
    ax_syn  = fig.add_axes(image_locations[idx * 2 + 1], projection=cutout_synth.wcs)

    cmap = plt.get_cmap("viridis").copy()
    cmap.set_under("black")
    cmap.set_over("white")

    im0 = ax_real.imshow(real_data,  origin="lower", cmap=cmap, norm=norm_real)
    im1 = ax_syn.imshow(synth_data,  origin="lower", cmap=cmap, norm=norm_synth)
    
    ax_real.coords.grid(color='white', linestyle='--', linewidth=1, alpha=0.7)
    ax_syn.coords.grid(color='white', linestyle='--', linewidth=1, alpha=0.7)

    cbar0 = plt.colorbar(im0, ax=ax_real, fraction=0.046, pad=0.04)
    cbar0.set_label("W/m2/Hz/sr", fontsize=fontsize_lg)
    cbar0.ax.tick_params(labelsize=fontsize_sm)

    cbar1 = plt.colorbar(im1, ax=ax_syn, fraction=0.046, pad=0.04)
    cbar1.set_label("W/m2/Hz/sr", fontsize=fontsize_lg)
    cbar1.ax.tick_params(labelsize=fontsize_sm)

    ax_real.add_patch(Circle((x_r, y_r), real_radius_pix,  edgecolor='red', facecolor='none', linewidth=2))
    ax_syn.add_patch( Circle((x_s, y_s), synth_radius_pix, edgecolor='red', facecolor='none', linewidth=2))

    ax_real.set_xticks([])
    ax_real.set_yticks([])
    ax_syn.set_xticks([])
    ax_syn.set_yticks([])

ymin, ymax = ax_scat.get_ylim()
text_y_pos = ymin * 1.1
ax_scat.axhline(y=1, color='gray', linestyle='--', linewidth=4, alpha=0.5)
ax_scat.axvline(x=5e-6, color='gray', linestyle='--', linewidth=4, alpha=0.7)
ax_scat.text(4.9e-6, text_y_pos, "← NIRCam",
                ha='right', va='center',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
                fontsize=fontsize_lg)

# Add MIRI label to the right
ax_scat.text(5.1e-6, text_y_pos, "MIRI →",
                ha='left', va='center',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
                fontsize=fontsize_lg)
ax_spec.legend(loc='upper right', fontsize=fontsize_lg)

plt.show()


In [ ]:
loc0.files.keys()

In [ ]:
align_cube_to_image(
    images,
    'F150W',
    loc0,
    'ch1ch2',
    'F150W',
    out_name = 'test',
    out_file = None,
    fwhm = 2.3,
    detection_threshold = 5.0,
    max_offset_pixels = 10.0,
    min_matches = 3,
)

In [ ]:
loc0.cubes['test'].header['BUNIT']

In [ ]:
align_cube_to_image(
    images,
    'F150W',
    loc0,
    'ch1ch2',
    "F150W",
    out_name = 'test_150',
    out_file = None,
    fwhm = 2.5,
    detection_threshold = 5.0,
    max_offset_pixels = 10.0,
    min_matches = 3,
    show_diagnostic_plots = True,
)
align_cube_to_image(
    images,
    'F187N',
    loc0,
    'ch1ch2',
    "F187N",
    out_name = 'test_187',
    out_file = None,
    fwhm = 2.5,
    detection_threshold = 5.0,
    max_offset_pixels = 10.0,
    min_matches = 3,
    show_diagnostic_plots = True,
)


loc0.create_synthetic_image('test_150', 'F150W', warnings=True, counter='energy', out_name='test_150')
loc0.create_synthetic_image('test_187', 'F187N', warnings=True, counter='energy', out_name='test_187')
loc0.display(['test_150', 'test_187'], locations[0], 0.75*u.arcsec, ncols=3, cmap='viridis', zoom=5, show_grid=True)

In [ ]:
loc0.images['synthetic_F150W']

In [ ]:
loc0.cubes['ch1']

In [ ]:
#TJ this was previous work, probably not useful


import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re
import sys
from astropy.io import fits
from astropy.visualization import simple_norm, imshow_norm
from ipywidgets import interact, Dropdown
from astropy.wcs import WCS
from astropy.constants import c
from photutils.aperture import CircularAperture, aperture_photometry
import astropy.units as u
from astropy.table import Table
from tabulate import tabulate
from pathlib import Path

parent_dir = Path().resolve().parent #TJ current notebook's parent directory

os.chdir(parent_dir) #TJ change working directory to be the parent directory
from Py_files.Basic_analysis import * #import basic functions from custom package


def import_data_and_sort_by_wavelength(file_path):
    '''import data and sort by wavelength
    -------------
    Parameters
    -------------
    file_path : type = str - path to file with data
    
    Returns
    -------------
    structured array ('wavelength', 'intensity', 'uncertainty', 'mask', 'note')
    '''    
    with open(file_path, 'r') as file:
        header = file.readline().strip().split()
        if ((len(header) == 3) & (type(try_float(header[0])) == type(0.1)) & (type(try_float(header[1])) == type(0.1)) & (type(try_float(header[2])) == type(0.1))):
            data_list = []
            data_list.append((try_float(header[0])*1e-6, try_float(header[1])*1e-20, try_float(header[2])*1e-20))
            for line in file:
                parts = line.strip().split(maxsplit=3)
                
                # Convert only numeric columns to floats
                wavelength = float(parts[0])*1e-6 #TJ required for sorting
                intensity = try_float(parts[1])*1e-20
                uncertainty = try_float(parts[2])*1e-20
                
                data_list.append((wavelength, intensity, uncertainty))
        
            # Define dtype with notes as string
            dtype = [
                ('wavelength', float),
                ('intensity', float),
                ('uncertainty', float),
            ]
            
            data = np.array(data_list, dtype=dtype)
            sorted_data = np.sort(data, order=['wavelength'])  # Sort by wavelength
            
            return sorted_data
        elif len(header) == 5:
            data_list = []
            for line in file:
                parts = line.strip().split(maxsplit=4)
                
                # Ensure exactly 5 parts (pad missing values with empty strings)
                parts = parts + [''] * (5 - len(parts))
                
                # Convert only numeric columns to floats
                wavelength = float(parts[0])  # Required for sorting
                intensity = try_float(parts[1])
                uncertainty = try_float(parts[2])
                mask = try_float(parts[3])
                notes = parts[4].strip()  # Keep as string, strip whitespace
                
                data_list.append((wavelength, intensity, uncertainty, mask, notes))
        
        # Define dtype with notes as string
        dtype = [
            ('wavelength', float),
            ('intensity', float),
            ('uncertainty', float),
            ('mask', float),
            ('notes', 'U256')  # Unicode string (max length 256)
        ]
        
        data = np.array(data_list, dtype=dtype)
        sorted_data = np.sort(data, order=['wavelength'])  # Sort by wavelength
        
        return sorted_data




# Step 1: Extract filter names (e.g., "f164n" from .fits, "F070M" from .dat)
def extract_filter_name(filename):
    # For .fits files: ngc5194_nircam_1v3_f164n_i2d.fits → "f164n"
    if filename.endswith('.fits'):
        parts = os.path.basename(filename).split('_')
        for part in parts:
            if part.startswith('f') and part[1:].replace('n', '').replace('w', '').replace('m', '').isdigit():
                return part.lower()
    # For .dat files: F070M.dat → "f070m"
    elif filename.endswith('.dat'):
        return os.path.splitext(os.path.basename(filename))[0].lower()
    return None


def generate_list_of_files():
        
    filter_directory = '/d/crow1/tools/cigale/database_builder/filters/jwst/'
    path = ['nircam', 'miri']
    filter_files = np.concatenate([glob.glob(os.path.join(filter_directory + file_path, "*.dat")) for file_path in path])
    image_directory = 'Data_files/Image_files'
    image_files = glob.glob(os.path.join(image_directory, "*.fits"))
    # Initialize aligned lists
    image_file_array = []
    filter_file_array = []
    
    # Loop through .fits files and find matching .dat files
    for fits_file in image_files:
        fits_filter = extract_filter_name(fits_file)
        if not fits_filter:
            continue  # Skip if no filter name found
        
        # Search for matching .dat file
        for dat_file in filter_files:
            dat_filter = extract_filter_name(dat_file)
            if dat_filter == fits_filter:
                image_file_array.append(fits_file)
                filter_file_array.append(dat_file)
                break  # Stop searching after first match
    return image_file_array, filter_file_array



def get_Fnu_transmission(Fnu_array, wl_array, transmission_array, trans_wl_array):
    '''get expected flux through filter in units of whatever the flux_array is. Make sure to convert to mks units
    -------------
    Parameters
    -------------
    Fnu_array : type = array - array of flux density values
    wl_array : type = array - array of wavelength values for the corresponding Fnu_array values
    transmission_array : type = array - array of unitless transmission coefficient
    trans_wl_array : type = array - array of wavelength values for the corresponding transmission values

    
    Returns
    -------------
    total_flux : type = float - in units of flux_array
    '''   
    Fnu_array = np.array(Fnu_array)
    wl_array = np.array(wl_array)
    transmission_array = np.array(transmission_array)
    trans_wl_array = np.array(trans_wl_array)
    # Convert wavelength to frequency, reverse so freq increases left to right
    spec_freq_array = c / wl_array[::-1]
    Fnu_array = Fnu_array[::-1]
    
    trans_freq_array = c / trans_wl_array[::-1]
    transmission_array = transmission_array[::-1]
    
    # Interpolate Fnu onto the transmission frequency grid
    interp_Fnu = np.interp(trans_freq_array, spec_freq_array, Fnu_array)
    weight = transmission_array / trans_freq_array
    numerator = np.trapz(interp_Fnu * weight, trans_freq_array)
    denominator = np.trapz(weight, trans_freq_array)
    ab_mean_flux = numerator / denominator
    # Numerator: Fν * Transmission integrated over frequency
    
    return ab_mean_flux


def get_Flambda_transmission(Fnu_array, wl_array, transmission_array, trans_wl_array):
    '''get expected flux through filter in units of whatever the flux_array is. Make sure to convert to mks units
    -------------
    Parameters
    -------------
    flux_array : type = array - array of flux values
    spec_wl_array : type = array - array of wavelength values for the corresponding flux_array values
    transmission_array : type = array - array of unitless transmission coefficient
    transmission_wl_array : type = array - array of wavelength values for the corresponding transmission values


    
    Returns
    -------------
    total_flux : type = float - in units of flux_array
    '''   
    first_wl = trans_wl_array[0]
    last_wl = trans_wl_array[-1]
    relevent_SED_wl = wl_array[((wl_array > first_wl) & (wl_array < last_wl))]
    relevent_SED_flux = Fnu_array[((wl_array > first_wl) & (wl_array < last_wl))]
    T_interp = np.interp(relevent_SED_wl, trans_wl_array, transmission_array)
    F_T = relevent_SED_flux * T_interp
    numerator = np.trapz(F_T, relevent_SED_wl)
    denominator = np.trapz(transmission_array, trans_wl_array)
    return numerator/denominator


In [ ]:
image_files, filter_files = generate_list_of_files()
SED_filepath = 'Data_files/ARM2_HII2_conv_stitched_test.dat' #TJ switch to this one and rerun to get the convolved array
#SED_filepath = 'Data_files/ARM2_HII2_stitch.dat'
SED_data = import_data_and_sort_by_wavelength(SED_filepath)
#SED_data = convert_sed_to_frequency(SED_data["wavelength"], SED_data["intensity"], SED_data["uncertainty"])
IFU_filepath = 'Data_files/IFU_files/M51_SW_f290lp_g395m-f290lp_s3d.fits'
hdul = fits.open(IFU_filepath)
IFU_data = hdul['SCI'].data  # flux in MJy/sr or μJy/arcsec²
IFU_header = hdul['SCI'].header
filter_data_array = []
radius = 0.75*u.arcsecond
aperture_area_sr = np.pi * (radius.to(u.rad))**2
for file in filter_files:
    data = []
    with open(file, 'r') as f:
            header = f.readline().strip().split()
            for line in f:
                data_line = line.strip().split()
                data.append(data_line)
            
    header, filter_T = data[:2], np.array(data[2:])
    wl = [try_float(filter_T[i,0])*1e-10 for i in range(len(filter_T))]
    T = [try_float(filter_T[i,1]) for i in range(len(filter_T))]
    #hz, T = convert_transmission_to_frequency(wl, T)
    filter_data_array.append([wl, T])




expected_flux_array = []

for i, filter_data in enumerate(filter_data_array):
    expected_flux_array.append((get_Fnu_transmission(SED_data["intensity"], SED_data["wavelength"], filter_data[1], filter_data[0])*aperture_area_sr).value)

    show_SED = False
    show_filter = False
    show_filter_pass = False
'''
    if show_filter:
        plt.plot(filter_data[0], filter_data[1], color = 'blue')
        plt.xlabel('wavelength (m)')
        plt.ylabel('T (%)')
        plt.title('filter transmission')
        plt.show()
    if show_SED:
        plt.plot(relevent_SED_wl, relevent_SED_flux, color = 'red')
        plt.xlabel('wavelength (m)')
        plt.ylabel('F*nu')
        plt.title('SED flux')
        plt.show()

    if show_filter_pass:
        plt.plot(relevent_SED_wl, F_T, color = 'purple')
        plt.xlabel('wavelength (m)')
        plt.ylabel('F*T*nu')
        plt.title(f'{filter_files[i].split("/")[-1]}')
        plt.show()
'''
print()


In [ ]:
#TJ on big beautiful cell for photometric

#TJ this cell gets the actual photometry data from Tony's files
# to get new files run at the command line:
#rclone copy m51_g_drive:/Data/GO3435/MIRI/Tony_reduction/v0p2/ /d/ret1/Taylor/jupyter_notebooks/Research/Data_files --include "*.fits"



'''
Draine info on regions, radius = 0.750 arcseconds
202.5062429  47.2143358  0.750  ARM1_HII
202.4335225  47.1729608  0.750  ARM2_HII1
202.4340450  47.1732517  0.750  ARM2_HII2 #TJ my region SW2 region
202.4823742  47.1958589  0.750  ARM3_HII
'''
# Define aperture (RA, Dec in degrees, radius in arcseconds)
arm1h_loc = [202.5062429, 47.2143358]
arm2h1_loc = [202.4335225, 47.1729608]
arm2h2_loc = [202.4340450, 47.1732517]
arm3h_loc = [202.4823742, 47.1958589]
#TJ m51 agn is at RA 13h 29m 53s | Dec +47° 11′
agn_loc = [(((13 + ((29/60) + (53/3600)))/24)*(360)), (47 + (11/60) + (43/3600))]
print(agn_loc[0], agn_loc[1],0)
filter_name_array = [f.split("/")[-1] for f in filter_files]
obs_flux_array = []
for i, file in enumerate(image_files):
    hdul = fits.open(file)
    data = hdul['SCI'].data*1e-20  # flux in MJy/sr or μJy/arcsec²
    header = hdul['SCI'].header
    exp_time = header['XPOSURE']
    pix_area = header["PIXAR_SR"]
    wcs = WCS(header)
    radius = 0.75 * u.arcsec
    radius_pixels = (radius).to_value(u.deg) / abs(header['CDELT2'])
    
    # Convert RA/Dec to pixel coordinates
    agn_x, agn_y = wcs.all_world2pix(agn_loc[0], agn_loc[1],0)
    x1, y1 = wcs.all_world2pix(arm1h_loc[0], arm1h_loc[1], 0)
    x2, y2 = wcs.all_world2pix(arm2h1_loc[0], arm2h1_loc[1], 0)
    x3, y3 = wcs.all_world2pix(arm2h2_loc[0], arm2h2_loc[1], 0)
    x4, y4 = wcs.all_world2pix(arm3h_loc[0], arm3h_loc[1], 0)
    aperture1 = CircularAperture((x1, y1), r=radius.to_value(u.deg) / header['CDELT2'])
    aperture2 = CircularAperture((x2, y2), r=radius.to_value(u.deg) / header['CDELT2'])
    aperture3 = CircularAperture((x3, y3), r=radius.to_value(u.deg) / header['CDELT2'])
    aperture4 = CircularAperture((x4, y4), r=radius.to_value(u.deg) / header['CDELT2'])
    agn_aperture = CircularAperture((agn_x, agn_y), r = (radius.to_value(u.deg) / header['CDELT2'])) 
    
    # Perform aperture photometry
    phot_result1 = aperture_photometry(data, aperture1)
    phot_result2 = aperture_photometry(data, aperture2)
    phot_result3 = aperture_photometry(data, aperture3)
    phot_result4 = aperture_photometry(data, aperture4)
    agn_phot = aperture_photometry(data,agn_aperture)
    total_flux1 = phot_result1['aperture_sum'][0]*pix_area  # in image units
    total_flux2 = phot_result2['aperture_sum'][0]*pix_area  # in image units
    total_flux3 = phot_result3['aperture_sum'][0]*pix_area  # in image units
    total_flux4 = phot_result4['aperture_sum'][0]*pix_area  # in image units
    
    total_agn_flux = agn_phot['aperture_sum'][0]*pix_area
    
    # Apply unit conversion if needed (e.g., MJy/sr → μJy)

    #print(f"Total flux in aperture 1: {total_flux1} W/m^2/Hz")
    #print(f"Total flux in aperture 2: {total_flux2} W/m^2/Hz")
    print(f"Total flux in aperture 3: {total_flux3}")
    print(f"expected {expected_flux_array[i]}, ratio : {expected_flux_array[i]/total_flux3}")
    
    #print(f"Total flux in aperture 4: {total_flux4} W/m^2/Hz")
    #print(f"total flux from AGN : {total_agn_flux} W/m^2/Hz")

    obs_flux_array.append(total_flux3)
    def show_image(file):
        plt.figure(figsize=(12, 10))  # Larger figure
    
        data = fits.getdata(file)
        norm = simple_norm(data, stretch='asinh', percent=96.9)
        plt.imshow(data, norm=norm, cmap='gray', origin='lower')
        # Mark aperture center with a red dot
        plt.scatter(x1, y1, color='red', s=50, label='Aperture Center', alpha = 0.01)  # s=size
        plt.scatter(x2, y2, color='orange', s=50, label='Aperture Center')  # s=size
        plt.scatter(x3, y3, color='green', s=50, label='Aperture Center')  # s=size
        plt.scatter(x4, y4, color='blue', s=50, label='Aperture Center')  # s=size
        plt.scatter(agn_x, agn_y, color='cyan', s=50, label='AGN')  # s=size
        
        # (Optional) Overlay the aperture circle
        aperture1 = CircularAperture((x1, y1), r=radius_pixels)
        aperture2 = CircularAperture((x2, y2), r=radius_pixels)
        aperture3 = CircularAperture((x3, y3), r=radius_pixels)
        aperture4 = CircularAperture((x4, y4), r=radius_pixels)
        agn_aperture = CircularAperture((agn_x, agn_y), r = radius_pixels)
        aperture1.plot(color='red', lw=2, alpha=0.1, label=f'{radius}" Aperture')
        aperture2.plot(color='orange', lw=2, alpha=0.7, label=f'{radius}" Aperture')
        aperture3.plot(color='green', lw=2, alpha=0.7, label=f'{radius}" Aperture')
        aperture4.plot(color='blue', lw=2, alpha=0.7, label=f'{radius}" Aperture')
        agn_aperture.plot(color = 'cyan', lw=2, alpha=0.7, label = 'AGN')
        plt.title(f'{file.split("lv3_")[-1]}')
        plt.show()
    show_image(file)
obs_flux_array = np.array(obs_flux_array)
expected_flux_array = np.array(expected_flux_array)

In [ ]:
expected_array = np.array([i for i in expected_flux_array])

observed_array = np.array(obs_flux_array)

# Extract the numerical part from filter names and get sorting indices
def get_filter_number(name):
    """
    Extract numbers between any two letters in a filter name.
    Examples:
        'F1130W' → 1130
        'F200W'  → 200
        'F560M'  → 560
        'NIRCam-F444W' → 444
    """
    match = re.search(r'[A-Za-z](\d+)[A-Za-z]', name)  # Numbers between ANY letters
    return int(match.group(1)) if match else 0

# Create array of numerical values for sorting
filter_numbers = np.array([get_filter_number(name) for name in filter_name_array])

# Get the sorting indices
sort_indices = np.argsort(filter_numbers)

# Apply sorting to all arrays
sorted_filter_names = np.array(filter_name_array)[sort_indices]
sorted_expected = np.array(expected_array)[sort_indices]

sorted_observed = np.array(observed_array)[sort_indices]
# Set a professional style at the beginning
plt.style.use('seaborn-v0_8-paper')  # Or try 'seaborn-whitegrid', 'ggplot'

# Use a larger font size for readability
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12})
# Now plot with sorted data
plt.figure(figsize=(10, 6))
plt.scatter([x.split('.')[0] for x in sorted_filter_names], sorted_expected/sorted_expected, label='Spectrum-derived', s=100, marker='o', color='blue')

plt.scatter([x.split('.')[0] for x in sorted_filter_names], sorted_observed/sorted_expected, label='Image-extracted', s=100, marker='x', color='red')

plt.xticks(rotation=45, ha='right')
plt.tick_params(axis='y', which='both', labelsize=10)
plt.legend()
plt.xlabel('Filter Names')
plt.ylabel('Filter Pass Through (MJy)')
plt.title("FLux transmitted through filter compared to spectrum-derived expectations \nNormalized to Spectrum-derived values")
plt.tight_layout()
plt.show()

# Set a professional style at the beginning
plt.style.use('seaborn-v0_8-paper')  # Or try 'seaborn-whitegrid', 'ggplot'

# Use a larger font size for readability
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12})
plt.figure(figsize=(10, 6))
plt.xticks(rotation=45, ha='right')

plt.scatter([x.split('.')[0] for x in sorted_filter_names], sorted_expected, label='Spectrum-derived', s=100, marker='o', color='blue')

plt.scatter([x.split('.')[0] for x in sorted_filter_names], sorted_observed, label='Image-extracted', s=100, marker='x', color='red')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.yscale('log')
plt.xlabel('Filter Names')
plt.ylabel('Filter Pass Through (MJy)')
plt.title('FLux transmitted through filter compared to spectrum-derived expectations')
plt.show()


In [ ]:
file = IFU_filepath
hdul = fits.open(file)
data = hdul['SCI'].data  # flux in MJy/sr or μJy/arcsec²
header = hdul['SCI'].header
header

In [ ]:
hdul = fits.open(image_files[0])
header = hdul['SCI'].header
header

In [ ]:
#TJ Now comparing model to model filter fluxes
def good(Fnu_array, wl_array, transmission_array, trans_wl_array):
    """
    Returns expected flux in mJy through a filter.
    All wavelength inputs are in meters. Fnu_array must be in mJy.
    """
    Fnu_array = np.array(Fnu_array)
    wl_array = np.array(wl_array)
    transmission_array = np.array(transmission_array)
    trans_wl_array = np.array(trans_wl_array)
    mask = [((wl_array >= trans_wl_array[0]) & (wl_array <= trans_wl_array[-1]))][0]
    
    # Convert wavelength to frequency, reverse so freq increases left to right
    spec_freq_array = c / wl_array[::-1]
    Fnu_array = Fnu_array[::-1]

    trans_freq_array = c / trans_wl_array[::-1]
    transmission_array = transmission_array[::-1]
    rel_spec_freq = spec_freq_array[((spec_freq_array > trans_freq_array[0]) & (spec_freq_array < trans_freq_array[-1]))]
    rel_Fnu = Fnu_array[((spec_freq_array > trans_freq_array[0]) & (spec_freq_array < trans_freq_array[-1]))]
    # Interpolate Fnu onto the transmission frequency grid
    interp_Fnu = np.interp(trans_freq_array, spec_freq_array, Fnu_array)
    weight = transmission_array / trans_freq_array
    numerator = np.trapz(interp_Fnu * weight, trans_freq_array)
    denominator = np.trapz(weight, trans_freq_array)

    T_interp = np.interp(rel_spec_freq, trans_freq_array, transmission_array)
    test_weight = T_interp / rel_spec_freq
    test_num = np.trapz(rel_Fnu*test_weight, rel_spec_freq)
    test_den = np.trapz(test_weight, rel_spec_freq)
    
    ab_mean_flux = numerator / denominator
    test_results = test_num/test_den
    # Numerator: Fν * Transmission integrated over frequency
    
    return ab_mean_flux, test_results

def test2(Fnu_array, wl_array, transmission_array, trans_wl_array):
    """
    Returns expected flux in mJy through a filter.
    All wavelength inputs are in meters. Fnu_array must be in mJy.
    """
    Fnu_array = np.array(Fnu_array)
    wl_array = np.array(wl_array)
    transmission_array = np.array(transmission_array)
    trans_wl_array = np.array(trans_wl_array)
    # Convert wavelength to frequency, reverse so freq increases left to right
    spec_freq_array = c / wl_array[::-1]
    Fnu_array = Fnu_array[::-1]

    trans_freq_array = c / trans_wl_array[::-1]
    transmission_array = transmission_array[::-1]

    # cross_match into observed frequencies
    closest_indices = np.argmin(np.abs(trans_freq_array[:, None] - spec_freq_array), axis=0)
    closest_T_array = transmission_array[closest_indices]
    
    interp_Fnu = np.interp(trans_freq_array, spec_freq_array, Fnu_array)
    weight = transmission_array / trans_freq_array
    numerator = np.trapz(interp_Fnu * weight, trans_freq_array)
    denominator = np.trapz(weight, trans_freq_array)
    ab_mean_flux = numerator / denominator
    # Numerator: Fν * Transmission integrated over frequency
    
    return ab_mean_flux

def predict_filter_band_pass(flux_array, spec_wl_array, transmission_array, transmission_wl_array):
    '''get expected flux through filter in units of whatever the flux_array is. Make sure to convert to mks units
    -------------
    Parameters
    -------------
    flux_array : type = array - array of flux values
    spec_wl_array : type = array - array of wavelength values for the corresponding flux_array values
    transmission_array : type = array - array of unitless transmission coefficient
    transmission_wl_array : type = array - array of wavelength values for the corresponding transmission values


    
    Returns
    -------------
    total_flux : type = float - in units of flux_array
    '''   

    
    
    flux_array = np.array(flux_array)
    spec_wl_array = np.array(spec_wl_array)
    transmission_array = np.array(transmission_array)
    transmission_wl_array = np.array(transmission_wl_array)
    #Try normalizing to max Transmission:
    max_t = np.max(transmission_array)
    
    
    first_wl = transmission_wl_array[0]
    last_wl = transmission_wl_array[-1]
    rel_wl_array = spec_wl_array[((spec_wl_array >= first_wl) & (spec_wl_array <= last_wl))]
    rel_flux_array = flux_array[((spec_wl_array >= first_wl) & (spec_wl_array <= last_wl))]
    
    closest_indices = np.argmin(np.abs(transmission_wl_array[:, None] - rel_wl_array), axis=0)
    closest_T_array = transmission_array[closest_indices]
    T_interp = np.interp(rel_wl_array, transmission_wl_array, transmission_array)
    
    interp_F_T = rel_flux_array * T_interp
    nk_F_T = rel_flux_array * closest_T_array
    
    interp_numerator = np.trapz(interp_F_T, rel_wl_array)
    nk_numerator = np.trapz(nk_F_T, rel_wl_array)
    
    denominator = np.trapz(transmission_array, transmission_wl_array)
    
    interp_flux = ((interp_numerator/denominator))
    nk_flux = ((nk_numerator/denominator))
    
    
    return interp_flux, nk_flux

def get_error(observed, expected):
    return ((observed-expected)/expected)*100

filter_flux_file_path =  '/d/vel2/ddale/phangs/synthetic/templates/models-block-0.fits'
filter_flux_file = fits.open(filter_flux_file_path)
filter_flux_tbl = Table.read(filter_flux_file[1])
table_data = []

spectral_file_path = '/d/vel2/ddale/phangs/synthetic/templates/00_best_model.fits'
spectral_file = fits.open(spectral_file_path)
spec_tbl = Table.read(spectral_file_path, hdu=1)
wl_array = spec_tbl['wavelength']*1e-9
Fnu_array = spec_tbl['Fnu']
keep = [23, 19, 12, 11, 10]

best = []
test = []
for filter_data in [filter_data_array[i] for i in keep]:
    trans_wl_array = filter_data[0]
    transmission_array = filter_data[1]
    results = good(Fnu_array, wl_array, transmission_array, trans_wl_array)
    best.append(results[0])
    test.append(results[1])



for i in range(5):
    filter_name = [filter_name_array[j] for j in keep][i]
    reported_value = filter_flux_tbl[0][filter_flux_tbl.colnames[i+1]]
    best_error = get_error(best[i], reported_value)
    test_error = get_error(test[i], reported_value)
    
    table_data.append([
        filter_name,
        f"{reported_value:.0f}",
        f"{best[i]:.2f}",
        f"{best_error:.4f}%",
        f"{test[i]:.2f}",
        f"{test_error:.4f}%"

        
    ])

# Print the table with headers
headers = [
    "Filter", 
    "Real_flux", 
    "best",
    "best_error",
    "test",
    "test error"
]
print('"test" trials are using interpolated transmission factors at intermediate wavelengths. \n"best" trials use interpolated flux values to get flux at the exact wavelength the transmission data is given for.')
print(tabulate(table_data, headers=headers, tablefmt="grid", floatfmt=".0f"))

In [ ]:
filter_flux_file_path =  '/d/vel2/ddale/phangs/synthetic/templates/models-block-0.fits'
filter_flux_file = fits.open(filter_flux_file_path)
filter_flux_tbl = Table.read(filter_flux_file[1])
filter_flux_tbl

In [ ]:
plt.plot(wl_array, Fnu_array)
plt.yscale('log')
plt.xscale('log')
plt.show()
spec_tbl

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)  # Higher DPI for publication

# Plot with error bars if you have them

plt.scatter([x.split('.')[0] for x in sorted_filter_names], 
            sorted_observed/sorted_expected, 
            label='Image-extracted', 
            s=100, marker='x', color='red', linewidth=1.5)

plt.xticks(rotation=45, ha='right')
plt.tick_params(axis='y', which='both', labelsize=14)
plt.tick_params(axis='x', which='both', labelsize=12)

plt.xlabel('Filter Names', fontweight='bold')
plt.ylabel('f_nu(image-extracted)/f_nu(spectrally-derived)', fontweight='bold')

# More professional title (consider moving to caption)
plt.title("FLux transmitted through filter compared to spectrum-derived expectations \nNormalized to Spectrum-derived values", pad=20)


# Adjust y-axis limits if needed
plt.ylim(0.9*min(sorted_observed/sorted_expected), 1.1*max(sorted_observed/sorted_expected))

plt.tight_layout()
plt.axvline(x=15.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
plt.axhline(y=1, color='gray', linestyle='--', linewidth=1, alpha=0.3)
ymin, ymax = plt.ylim()
# Place text at 90% of ymax
text_y_pos = ymax * 0.95

# Add NIRCam label to the left
plt.text(15.25, text_y_pos, "← NIRCam", 
         ha='right', va='center', 
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
         fontsize=10)

# Add MIRI label to the right
plt.text(15.75, text_y_pos, "MIRI →", 
         ha='left', va='center', 
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
         fontsize=10)

plt.savefig('flux_comparison_normalized.png', bbox_inches='tight', transparent=False)
plt.show()





plt.figure(figsize=(10, 6), dpi=300)

# Use consistent color scheme with first plot
plt.scatter([x.split('.')[0] for x in sorted_filter_names], 
            sorted_expected, 
            label='Spectrum-derived', 
            s=100, marker='o', color='blue', edgecolor='black', linewidth=0.5)

plt.scatter([x.split('.')[0] for x in sorted_filter_names], 
            sorted_observed, 
            label='Image-extracted', 
            s=100, marker='x', color='red', linewidth=1.5)

plt.xticks(rotation=45, ha='right')
plt.tick_params(axis='y', which='both', labelsize=14)
plt.tick_params(axis='x', which='both', labelsize=12)
plt.legend(frameon=True, framealpha=1, loc='best')
plt.yscale('log')
plt.xlabel('Filter Names', fontweight='bold')
plt.ylabel('Flux density \n$(W/m^2/Hz)$', fontweight='bold')  # Verify units
plt.title("FLux transmitted through filter compared to spectrum-derived expectations", pad=20)

plt.axvline(x=15.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ymin, ymax = plt.ylim()
# Place text at 90% of ymax
text_y_pos = ymax * 0.7

# Add NIRCam label to the left
plt.text(15.25, text_y_pos, "← NIRCam", 
         ha='right', va='center', 
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
         fontsize=10)

# Add MIRI label to the right
plt.text(15.75, text_y_pos, "MIRI →", 
         ha='left', va='center', 
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2),
         fontsize=10)
plt.tight_layout()
plt.savefig('flux_comparison.png', bbox_inches='tight', transparent=False)
plt.show()